The following has been adapted from Hailo's DFC Tutorials 1 and 2 (Parsing and Optimizing with DFC). It was run from a docker container setup using Hailo AI SoftwareSuite


In [ ]:
# General imports used throughout the tutorial
# file operations
import json
import os

import numpy as np
import tensorflow as tf
from IPython.display import SVG
from matplotlib import patches
from matplotlib import pyplot as plt
from PIL import Image
from tensorflow.python.eager.context import eager_mode

import torchvision as tv
import torch

import cv2

# import the hailo sdk client relevant classes
from hailo_sdk_client import ClientRunner, InferenceContext

%matplotlib inline

IMAGES_TO_VISUALIZE = 5

In [ ]:
chosen_hw_arch = "hailo8"

The ONNX model below was created using the `ultralytics` yolo11n.pt pretrained model, which was finetuned on a customised visdrone dataset (single class) and exported using:
```model.export(format='onnx', opset=14)```
    
the onnx output model was copied to the docker container into /local/shared_with_docker/yolov11n_visdrone.onnx
    
 

In [ ]:
onnx_model_name = "yolo11n_visdrone"
onnx_path = "/local/shared_with_docker/yolo11n_visdrone_2class.onnx"

The endnodes below were taken from the [yolov11n.yaml](https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/networks/yolov11n.yaml) on the hailo_model_zoo github. The finetuned (on VisDroneCustomClass) onnx model was opened in netron to double check these end nodes
```
- /model.23/cv2.0/cv2.0.2/Conv
- /model.23/cv3.0/cv3.0.2/Conv
- /model.23/cv2.1/cv2.1.2/Conv
- /model.23/cv3.1/cv3.1.2/Conv
- /model.23/cv2.2/cv2.2.2/Conv
- /model.23/cv3.2/cv3.2.2/Conv
```

In [ ]:
runner = ClientRunner(hw_arch=chosen_hw_arch)
hn, npz = runner.translate_onnx_model(
    onnx_path,
    onnx_model_name,
    start_node_names=["/model.0/conv/Conv"],
    end_node_names=["/model.23/cv2.0/cv2.0.2/Conv", 
                   "/model.23/cv3.0/cv3.0.2/Conv",
                   "/model.23/cv2.1/cv2.1.2/Conv",
                   "/model.23/cv3.1/cv3.1.2/Conv",
                   "/model.23/cv2.2/cv2.2.2/Conv",
                   "/model.23/cv3.2/cv3.2.2/Conv"],
    net_input_shapes={"/model.0/conv/Conv": [1, 3, 640, 640]},
)

In [ ]:
#save the parsed model
hailo_model_har_name = f"{onnx_model_name}_hailo_model_op14.har"
runner.save_har(hailo_model_har_name)

In [ ]:
def preproc(image, output_height=640, output_width=640):
    preprocess = tv.transforms.Compose([
        tv.transforms.Resize((output_height, output_width)),
    ])
    
    data = np.array(preprocess(image))
    
    return data

In [ ]:
!ls -la ../data/visdrone/val/images

In [ ]:
annotations_file = '/local/shared_with_docker/visdrone/annotations_VisDroneHumans_val.json'
images_path = "/local/shared_with_docker/VisDrone2019-DET-val/images"

with open(annotations_file, 'r') as f:
    val_gt = json.load(f)
    f.close()
    
images = val_gt['images']
image_list = [(x['file_name'],x['id']) for x in images] 


dataset_sz = len(image_list)
val_dataset = np.zeros((dataset_sz, 640, 640, 3))
val_imageids = np.zeros((dataset_sz,1))
for idx, imagename_id in enumerate(image_list):
    imgname, imgid = imagename_id
    image_file = os.path.join(images_path, imgname)
    if idx==dataset_sz:
        break
    img = Image.open(image_file).convert('RGB')
    img_preproc = preproc(img)
    val_dataset[idx, :, :, :] = img_preproc
    val_imageids[idx] = imgid


In [ ]:
# quantization needs calibration data, use the training data for this
cal_images_path = "../data/visdrone/train/images/" # use training data for calib
cal_images_list = [img_name for img_name in os.listdir(cal_images_path) if os.path.splitext(img_name)[1] == ".jpg"]
dataset_sz = 4000 
calib_dataset = np.zeros((dataset_sz, 640, 640, 3))

for idx, img_name in enumerate(sorted(cal_images_list)):
    if idx==dataset_sz:
        break
    img = Image.open(os.path.join(images_path, img_name)).convert('RGB')
    img_preproc = preproc(img)
    calib_dataset[idx, :, :, :] = img_preproc

In [ ]:
print(f"Calibration dataset size: {calib_dataset.shape}\
        \nValidation dataset shape: {val_dataset.shape}")


In [ ]:
#load our parsed HAR from the Parsing Tutorial
assert os.path.isfile(hailo_model_har_name), "Please provide valid path for HAR file"
runner = ClientRunner(har=hailo_model_har_name)

The following was taken from trieut415's [guide for parsing/compiling a cuatom yolov11 model on colab](https://community.hailo.ai/t/guide-to-using-the-dfc-to-convert-a-modified-yolov11-on-google-colab/7131)

In [ ]:
from pprint import pprint

try:
    # Access the HailoNet as an OrderedDict
    hn_dict = runner.get_hn()  # Or use runner._hn if get_hn() is unavailable
    print("Inspecting layers from HailoNet (OrderedDict):")

    # Pretty-print each layer
    for key, value in hn_dict.items():
        print(f"Key: {key}")
        pprint(value)
        print("\n" + "="*80 + "\n")  # Add a separator between layers for clarity

except Exception as e:
    print(f"Error while inspecting hn_dict: {e}")

In [ ]:
# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)
alls =  """
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
change_output_activation(conv54, sigmoid)
change_output_activation(conv65, sigmoid)
change_output_activation(conv80, sigmoid)
nms_postprocess("../yolov11_nms_config_visdrone.json", meta_arch=yolov8, engine=cpu)

allocator_param(width_splitter_defuse=disabled)
 """
#model_optimization_config(calibration, batch_size=16, calibset_size=1024)
#post_quantization_optimization(finetune, policy=enabled, learning_rate=0.0001, epochs=8, dataset_size=2900)

# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(alls)
#runner.optimize_full_precision()
runner.optimize(calib_dataset)

In [ ]:
model_name = "yolo11n_visdrone"
# fp_model_har_path = f"{model_name}_fp_opt_model_visdrone.har"
# runner.save_har(fp_model_har_path)

quant_model_har_path = f"{model_name}_quant_opt_model_visdrone.har"
runner.save_har(quant_model_har_path)

In [ ]:
# load the Quantized HAR file
# fp_model_har_path = f"{model_name}_fp_opt_model_visdrone.har"
# runner = ClientRunner(har=fp_model_har_path, hw_arch=chosen_hw_arch)

quant_model_har_path = f"{model_name}_quant_opt_model_visdrone.har"
runner = ClientRunner(har=quant_model_har_path, hw_arch=chosen_hw_arch)


In [ ]:
# # Call Optimize to perform the optimization process
# runner.optimize(calib_dataset)

# # Save the result state to a Quantized HAR file
# quantized_model_har_path = f"{model_name}_quantized_model_visdrone.har"
# runner.save_har(quantized_model_har_path)

In [ ]:
sample_dataset = np.zeros((2, 640, 640, 3))
SAMPLE_IMAGE_PATH = '../data/visdrone/val/images/0000001_03999_d_0000007.jpg'
img = Image.open(SAMPLE_IMAGE_PATH).convert('RGB')
img_preproc = preproc(img)
sample_dataset[0,:,:,:] = img_preproc

# Notice that we use the original images, because normalization is IN the model
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    output = runner.infer(ctx, sample_dataset[:1, :, :, :])

In [ ]:


# Notice that we use the original images, because normalization is IN the model
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    quant_outputs = runner.infer(ctx, val_dataset)

In [393]:
offset = 0
cutoff = len(val_dataset)
output = quant_outputs[offset:offset+cutoff]

img_dim_map = {x["id"]: (x["width"], x["height"]) for x in images}

output.shape


(532, 2, 5, 100)

In [394]:

# remove zero-padding and transpose detections into single array
combined = np.empty((9, 0)) # 1 classid, xywh, score
for j in range(output.shape[0]):
    imgid = int(val_imageids[j+offset][0])
    w, h = img_dim_map[imgid]
    for i in range(output.shape[1]):
        print(f"processing class {i} for image {imgid}...")
        valid_mask = np.any(output[j,i,:,:] != 0, axis=(0))  # Check if any non-zero values exist in each column
        last_valid_indices = np.argmax(~valid_mask, axis=0) # First occurrence of zero padding
        dets = output[j, i, :, :last_valid_indices]
        
        #add extra cols
        class_col = np.full_like(dets[0,None], i) # add the classID into the array
        imgid_col = np.full_like(dets[0,None], imgid)
        w_col = np.full((1, dets.shape[1]), w)
        h_col = np.full((1, dets.shape[1]), h)
        
        dets = np.vstack((dets, class_col, imgid_col, w_col, h_col))
        combined = np.concatenate((combined, dets), axis=1)
    
    
dets = combined.T
dets
# # remove all non-zero padding elements
# valid_mask = np.any(output != 0, axis=(1, 2))  # Check if any non-zero values exist in each column
# last_valid_indices = np.argmax(~valid_mask, axis=1) # First occurrence of zero padding
# last_valid_indices[valid_mask[:, -1]] = output.shape[-1] # edge case, no detections
# dets = output[0, 0, :, :last_valid_indices[0]]

#dets = dets.transpose()



processing class 0 for image 1...
processing class 1 for image 1...
processing class 0 for image 2...
processing class 1 for image 2...
processing class 0 for image 3...
processing class 1 for image 3...
processing class 0 for image 4...
processing class 1 for image 4...
processing class 0 for image 5...
processing class 1 for image 5...
processing class 0 for image 6...
processing class 1 for image 6...
processing class 0 for image 7...
processing class 1 for image 7...
processing class 0 for image 8...
processing class 1 for image 8...
processing class 0 for image 9...
processing class 1 for image 9...
processing class 0 for image 10...
processing class 1 for image 10...
processing class 0 for image 11...
processing class 1 for image 11...
processing class 0 for image 12...
processing class 1 for image 12...
processing class 0 for image 13...
processing class 1 for image 13...
processing class 0 for image 14...
processing class 1 for image 14...
processing class 0 for image 15...
pro

array([[    0.33515,     0.62828,     0.36738, ...,           1,        1360,         765],
       [    0.21245,     0.55237,     0.24305, ...,           1,        1360,         765],
       [      0.167,     0.54864,     0.19268, ...,           1,        1360,         765],
       ...,
       [    0.51258,     0.54986,     0.55202, ...,         532,         960,         540],
       [    0.31103,     0.43499,     0.33272, ...,         532,         960,         540],
       [    0.32735,      0.4149,     0.34862, ...,         532,         960,         540]])

In [ ]:
# multiply by input img size to denormalise xyxy
imgsz = np.array([640])
mul = np.multiply
buffer = np.empty_like(dets[:,:4])
imgsz_float = float(imgsz[0])

# the output is actually yxyx format
bbox = mul(imgsz_float, dets[:,:4], buffer)
bbox[:, [0, 1, 2, 3]] = bbox[:, [1, 0, 3, 2]]


xywh = xyxy2xywh(bbox)

results = np.hstack((xywh, dets[:,4:])) # add the score, class and imgid cols back in
print(results)
       



In [395]:
# def ltxy2xywh(xywh):
#     x1, y1, x2, y2 = xywh
#     x = x1
#     y = y1
#     w = x2 - x1
#     h = y2 - y1

#     return x1, y1, w, h
def ltxy2xywh(xywh):
    
    xywh[:,0] = xywh[:,0] # x
    xywh[:,1] = xywh[:,1] # y
    xywh[:,2] = xywh[:,2] - xywh[:,0] # x2 - x1
    xywh[:,3] = xywh[:,3] - xywh[:,1] # y2 - y1

    return xywh


print(f"dets:\n {dets[17]}")
xywh_list = np.full_like(dets, dets)
print(f"before swapping yx for xy:\n {xywh_list[17]}")
xywh_list[:, [0, 1,2,3]] = xywh_list[:, [1, 0, 3, 2]] # annoyingly, it is in yx, ont xy, what misery
print(f"after swapping yx for xy:\n {xywh_list[17]}")

xywh_list[:,0] = xywh_list[:,0] * xywh_list[:,7]
xywh_list[:,1] = xywh_list[:,1] * xywh_list[:,8]

xywh_list[:,2] = xywh_list[:,2] * xywh_list[:,7]
xywh_list[:,3] = xywh_list[:,3] * xywh_list[:,8]
print(f"before conversion:\n {xywh_list[17]}")
xywh = ltxy2xywh(xywh_list[:,:4])
xywh_list[:, :4] = xywh
print(f"after conversion:\n {xywh_list[17]}")
dets_scaled = xywh_list



dets:
 [    0.41794     0.43869     0.43638     0.44765     0.51593           1           2        1920        1080]
before swapping yx for xy:
 [    0.41794     0.43869     0.43638     0.44765     0.51593           1           2        1920        1080]
after swapping yx for xy:
 [    0.43869     0.41794     0.44765     0.43638     0.51593           1           2        1920        1080]
before conversion:
 [     842.29      451.37      859.48      471.29     0.51593           1           2        1920        1080]
after conversion:
 [     842.29      451.37      17.191      19.917     0.51593           1           2        1920        1080]


In [ ]:
# Load the image
SAMPLE_IMAGE_PATH = "../data/visdrone/val/images/0000001_03999_d_0000007.jpg"
imgpth = SAMPLE_IMAGE_PATH
img = cv2.imread(imgpth)

labpth = imgpth.replace("images", "labels").replace('jpg', 'txt')

# Get image dimensions
height, width, _ = img.shape
# Scaling factors
scale_x = width / 640
scale_y = height / 640

xywh_list = np.full_like(dets, dets)
for idx, el in enumerate(bbox):
# Rescale the coordinates
    x1_orig = int(el[0] * scale_x)
    y1_orig = int(el[1] * scale_y)
    x2_orig = int(el[2] * scale_x)
    y2_orig = int(el[3] * scale_y)

    xywh = ltxy2xywh([x1_orig, y1_orig, x2_orig, y2_orig])
    xywh_list[idx, :4] = xywh
    

dets_scaled = xywh_list
# xyxy2xywh_scaled = xyxy2xywh(np.array(dets_scaled))
# ltwh2xywh_scaled = ltwh2xywh(np.array([x1,y1,w,h]))
# ltxy2wh_scaled = ltxy2xywh(np.array([x1,y1,x2,y2]))
print(f"\
    original: {dets_scaled[0]}\n\
    xyxy2xywh: {xyxy2xywh_scaled[0]}\n\
    ltwh2xywh: {ltwh2xywh_scaled}\n\
    ltxy2wh: {ltxy2wh_scaled}")

dets_scaled

# swap the x1 x1




In [ ]:
# format detections into json format for COOC Evaluation


"""
TODO:
    Map image_id for results to imgage_id in annotation file
    remap classID (0 in model to 1 in ann file)
        run above with a single image through faster-COCO-Eval
    iterate model over full validation dataset and add output to json in above format
        run above for full result set on faster-COCO-Eval
    
"""
from ultralytics.utils.ops import (
    ltwh2xywh,
    ltwh2xyxy,
    xywh2ltwh,  # xywh → top-left corner, w, h
    xywh2xyxy,
    xywhn2xyxy,  # normalized → pixel
    xyxy2ltwh,  # xyxy → top-left corner, w, h
    xyxy2xywhn,  # pixel → normalized
)
xyxy =   np.array([1343.56787,861.94434,1367.06543, 909.77515])
xyxy = np.array(xyxy)
xyxy2xywh(xyxy)

### Map image_id for results to imgage_id in annotation file


In [396]:
def get_image_id(images, image_name):
    image_id = None
    for image in images:
        if image['file_name'] == image_name:
            image_id = image['id']
            return image_id
    assert image_id != None, f"Image {image_name} has not been found in list of images"
    if not image_id:
        raise
        
# SAMPLE_IMAGE_PATH = "../data/visdrone/val/images/0000001_03999_d_0000007.jpg"
# ann_file = "/local/shared_with_docker/visdrone/annotations_VisDrone_val.json"
# with open(ann_file, 'r') as file:
#     visdrone_val = json.load(file)
#     file.close()
    
# image_name = os.path.basename(SAMPLE_IMAGE_PATH)
# images = visdrone_val['images']
# image_id = get_image_id(images, image_name)

class_remap = {0: 1, 1: 2}
res = []
annId = 1
for el in dets_scaled:
    x,y,w,h,conf,classid, imgid,_,_ = el
    imgid = int(imgid)
    res.append({"image_id": imgid, "category_id": class_remap[classid], "bbox": [x,y,w,h], "score": conf, "id": annId, "segmentation": []}) 
    print({"image_id": imgid, "category_id": class_remap[classid], "bbox": [x,y,w,h], "score": conf, "id": annId, "segmentation": []}, '\n')
    annId +=1

outputfile = f"/local/shared_with_docker/visdrone/{image_name.split('.')[0]}_VisDrone_quant_dets.json" 
with open(outputfile, 'w') as outf:
    json.dump(res, outf)
    outf.close()
    
print(f"Total of {annId} annotations saved to {outputfile}")


{'image_id': 1, 'category_id': 1, 'bbox': [854.4633722305298, 256.39128014445305, 16.688380241394043, 24.6554633975029], 'score': 0.5960948467254639, 'id': 1, 'segmentation': []} 

{'image_id': 1, 'category_id': 1, 'bbox': [751.2207841873169, 162.5249321758747, 12.16242790222168, 23.41033101081848], 'score': 0.5628290176391602, 'id': 2, 'segmentation': []} 

{'image_id': 1, 'category_id': 1, 'bbox': [746.1443376541138, 127.75324121117592, 13.19556713104248, 19.6488406509161], 'score': 0.5127997398376465, 'id': 3, 'segmentation': []} 

{'image_id': 1, 'category_id': 1, 'bbox': [900.942964553833, 207.72083938121796, 17.17637538909912, 27.803336083889008], 'score': 0.5002924203872681, 'id': 4, 'segmentation': []} 

{'image_id': 1, 'category_id': 1, 'bbox': [840.5516195297241, 254.7434301674366, 16.23645782470703, 27.62422889471054], 'score': 0.43150222301483154, 'id': 5, 'segmentation': []} 

{'image_id': 1, 'category_id': 1, 'bbox': [754.5380163192749, 88.82427882403135, 13.8404989242553

In [ ]:
pred = [
    {"image_id": 4, "category_id": 1, "bbox": [1319, 861, 1344, 908], "score": 0.70283, "id": 1, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [621, 950, 645, 981], "score": 0.8573, "id": 2, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [1060, 533, 1082, 563], "score": 0.86297, "id": 3, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [998, 720, 1017, 742], "score": 0.95245, "id": 4, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [705, 213, 722, 236], "score": 0.91751, "id": 5, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [1031, 553, 1053, 579], "score": 0.74494, "id": 6, "segmentation": []},
    {"image_id": 4, "category_id": 1, "bbox": [1365, 485, 1385, 511], "score": 0.70283, "id": 7, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [777, 825, 799, 855], "score": 0.8573, "id": 8, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [981, 143, 993, 160], "score": 0.86297, "id": 9, "segmentation": []}, 
    {'image_id': 4, 'category_id': 1, 'bbox': [705, 215, 717, 239], 'score': 0.91251, 'id': 12, 'segmentation': []},     
    {"image_id": 4, "category_id": 1, "bbox": [1106, 534, 1125, 562], "score": 0.95245, "id": 10, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [962, 431, 981, 457], "score": 0.91751, "id": 11, "segmentation": []}, 
    {"image_id": 4, "category_id": 1, "bbox": [895, 263, 909, 285], "score": 0.74494, "id": 12, "segmentation": []}
]

outputfile = f"/local/shared_with_docker/visdrone/{image_name.split('.')[0]}_VisDrone_manual.json" 
with open(outputfile, "w") as f:
    json.dump(pred, f)

In [ ]:
with open("/local/shared_with_docker/visdrone/visdrone_imgId4_preds_ultralytics.json", 'r') as f:
    ul_out = json.load(f)
    f.close()

coco_format = []
# output is xyxy, convert to xywh, remap classid etc
for idx, detection in enumerate(ul_out):
    classid = int(detection['class']) +1 # convert back to coco
    name = detection["name"]
    score = detection["confidence"]
    x1 = detection["box"]['x1']
    y1 = detection["box"]['y1']
    x2 = detection["box"]['x2']
    y2 = detection["box"]['y2']
    xywh = xyxy2xywh(np.array([x1,y1,x2,y2]))
    x1,y2,w,h = [float(x) for x in xywh]
    coco_format.append({"image_id": 4, "category_id": classid, "category_name": name,  "bbox": [x1, x2, w, h], "score": int(score), "id": idx, "segmentation": []}) 
    
    
with open("/local/shared_with_docker/visdrone/visdrone_imgId4_preds_ultralytics_xywh.json", 'w') as f:
    json.dump(coco_format, f)
    f.close()

coco_format


In [ ]:




#Ground truth labels in YOLO format (class_id, x_center, y_center, bbox_width, bbox_height)
with open(labpth, "r") as f:
    gtlabels = [tuple(map(float, line.split())) for line in f]
    f.close()
    
print(gtlabels)
    


# Loop through labels and draw bounding boxes
for idx, labels in enumerate(gtlabels):
    class_id, x_center, y_center, bbox_width, bbox_height = labels

    # Convert normalized YOLO coordinates to pixel values
    x_center, y_center = int(x_center * width), int(y_center * height)
    bbox_width, bbox_height = int(bbox_width * width), int(bbox_height * height)

    # Calculate top-left and bottom-right corners
    x1_gt = int(x_center - bbox_width / 2)
    y1_gt = int(y_center - bbox_height / 2)
    x2_gt = int(x_center + bbox_width / 2)
    y2_gt = int(y_center + bbox_height / 2)
    print(f"ground truth: {[x1_gt, y1_gt, x2_gt, y2_gt]}")
    
    # Draw the bounding box
    colorgt = (0, 255, 0)  # Green color for bounding box
    colorprd = (150, 50, 100)  # Green color for bounding box

    thickness = 2
    cv2.rectangle(img, (x1_gt, y1_gt), (x2_gt, y2_gt), colorgt, thickness)

    # Add class label text
    cv2.putText(img, str(class_id), (x1_gt, y1_gt - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colorgt, 2)

    if idx < len(preds):
        print(f"predictions: {preds[idx][1:]}")
        clsid, x1_prd, y1_prd, x2_prd, y2_prd = preds[idx]
        cv2.rectangle(img, (x1_prd, y1_prd), (x2_prd, y2_prd), colorprd, thickness)

    


In [ ]:
# Display the image with bounding boxes
cv2.imshow("Ground Truth", img)
cv2.waitKey(0)  # Wait for key press
cv2.destroyAllWindows()  # Close window

In [ ]:

import cv2
import numpy as np

# Load the image
SAMPLE_IMAGE_PATH = '/media/citi-ai/matthew/uav-human-detection/hailo-ai/shared_with_docker/0000002_00005_d_0000014.jpg'
img = cv2.imread(SAMPLE_IMAGE_PATH)

# Get image dimensions
height, width, _ = img.shape
# Scaling factors
scale_x = width / 640
scale_y = height / 640

preds = []
for idx, el in enumerate(exp_labels):
# Rescale the coordinates
    x1_orig = int(el[0] * scale_x)
    y1_orig = int(el[1] * scale_y)
    x2_orig = int(el[2] * scale_x)
    y2_orig = int(el[3] * scale_y)
    print(f"Index: {idx}:\n{x1_orig}, {y1_orig}, {x2_orig}, {y2_orig}")
    preds.append([62, x1_orig, y1_orig, x2_orig, y2_orig])


# Ground truth labels in YOLO format (class_id, x_center, y_center, bbox_width, bbox_height)
gtlabels = [(0, 0.434896, 0.391667, 0.005208, 0.016667),
            (0, 0.450000, 0.376852, 0.008333, 0.024074),
            (0, 0.921875, 0.491667, 0.012500, 0.035185),
            (0, 0.938542, 0.432407, 0.008333, 0.031481),
            (0, 0.062500, 0.598148, 0.010417, 0.037037),
            (0, 0.245312, 0.450000, 0.007292, 0.025926),
            (0, 0.250000, 0.450000, 0.006250, 0.025926),
            (0, 0.066146, 0.402778, 0.009375, 0.024074),
            (0, 0.093750, 0.375000, 0.008333, 0.031481),
            (0, 0.231771, 0.227778, 0.007292, 0.018519),
            (0, 0.067187, 0.535185, 0.011458, 0.033333),
]


# Loop through labels and draw bounding boxes
for idx, labels in enumerate(gtlabels):
    class_id, x_center, y_center, bbox_width, bbox_height = labels

    # Convert normalized YOLO coordinates to pixel values
    x_center, y_center = int(x_center * width), int(y_center * height)
    bbox_width, bbox_height = int(bbox_width * width), int(bbox_height * height)

    # Calculate top-left and bottom-right corners
    x1_gt = int(x_center - bbox_width / 2)
    y1_gt = int(y_center - bbox_height / 2)
    x2_gt = int(x_center + bbox_width / 2)
    y2_gt = int(y_center + bbox_height / 2)
    print(f"ground truth: {[x1_gt, y1_gt, x2_gt, y2_gt]}")
    print(f"predictions: {preds[idx][1:]}")

    clsid, x1_prd, y1_prd, x2_prd, y2_prd = preds[idx]

    # Draw the bounding box
    colorgt = (0, 255, 0)  # Green color for bounding box
    colorprd = (150, 50, 100)  # Green color for bounding box

    thickness = 2
    cv2.rectangle(img, (x1_gt, y1_gt), (x2_gt, y2_gt), colorgt, thickness)
    cv2.rectangle(img, (x1_prd, y1_prd), (x2_prd, y2_prd), colorprd, thickness)

    # Add class label text
    cv2.putText(img, str(class_id), (x1_gt, y1_gt - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colorgt, 2)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import numpy as np
import pylab

import json
pylab.rcParams['figure.figsize'] = (10.0, 8.0)

annType = ['segm','bbox','keypoints']
annType = annType[1]      #specify type here
prefix = 'person_keypoints' if annType=='keypoints' else 'instances'
print('Running demo for *%s* results.'%(annType))

#initialize COCO ground truth api
dataDir='./'
dataType='val2017'
annFile = f"annotations/{prefix}_{dataType}.json"
annFile = '/local/shared_with_docker/visdrone/annotations_VisDrone_val.json'
cocoGt=COCO(annFile)

resFile = '/local/shared_with_docker/visdrone/visdrone_imgId4_preds_ultralytics_xywh.json'
cocoDt=cocoGt.loadRes(resFile)

imgIds=sorted(cocoGt.getImgIds())
imgIds= [4] 
catIDs = [1,2] # 81 for all IDs
maxDets = [1, 10, 100]

# running evaluation with legacy pycocotools
cocoEval = COCOeval(cocoGt,cocoDt,annType)
cocoEval.params.imgIds  = imgIds
cocoEval.params.catIds = catIDs
cocoEval.params.maxDets = maxDets
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

# import faster_coco_eval
# # Replace pycocotools with faster_coco_eval
# faster_coco_eval.init_as_pycocotools()

# from pycocotools.coco import COCO
# from pycocotools.cocoeval import COCOeval

# anno = COCO(str(annFile))  # init annotations api
# pred = anno.loadRes(str(resFile))  # init predictions api (must pass string, not Path)

# val = COCOeval(anno, pred, "bbox")
# val.params.imgIds  = imgIds
# val.params.maxDets = maxDets
# val.params.catIds = catIDs


# val.evaluate()
# val.accumulate()
# val.summarize()

In [ ]:
!pip install faster-coco-eval